# Prediccion de Default en Prestamos


Para este proyecto utilizaremos un sample de los datos de Lending Club. La idea es predecir si cierto usuario cometera Default basado en informacion que la plataforma recolecta. Esto nos ayudara a mejorar la metodologia/pipeline de prestamo.


# Descripcion



Contiene los prestamos de esta plataforma:

    periodo 2007-2017Q3.
    887mil observaciones, sample de 100mil
    150 variables
    Target: loan status



# Objetivo

Realizar un ETL y un EDA

## ETL

0. Limpia los datos de tal manera que al final del ETL queden en formato `tidy`.
1. Asegurate de cargar y leer los datos
2. Crea una tabla donde se guarde el nombre de la columna y el tipo de dato: (`column_name`,   `type`).
3. Asegurate de pensar cual es el tipo de dato correcto. Porque elejiste strig/object o float o int?. No hay respuestas incorrectas como tal, pero tienes que justificar tu decision.
4. Maneja missings o nans de la manera adecuada. Justifica cada decision







## EDA

0. Preparar lo datos para un pipeline de datos
1. Quitar columnas inservibles 
2. Imputar valores
3. Mantener replicabildiad y reproducibilidad

**No olvides anotar tus justificaciones en celdas para recordar cuando te toque explicarlo.** Puedes agregar el numero de celdas que necesites para poner tu explicacion y el codigo, solo manten la estructura.

# ETL

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import re
import json

Vas a obtener 2 errores, solucionalo con los visto en clase.  
Tip: Se arreglan con argumentos adicionales de la funcion `read_csv`  
Documentacion: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html 

In [ ]:
# Utilizamos el par?metro 'compression' con el valor 'gzip' para indicar que el archivo est? comprimido.
loans = pd.read_csv('https://github.com/sonder-art/fdd_prim_2023/blob/main/codigo/pandas/LoansData_sample.csv.gz?raw=true',compression='gzip')
# Mostrar el DataFrame cargado desde el archivo CSV
loans = loans.drop('Unnamed: 0',axis=1)
loans


## Tabla (column_name, type)

Revisa el metodo pd.DataFrame.dtypes. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html 

In [ ]:
# Obtener los tipos de datos de las columnas en el DataFrame 'loans'
column_types = loans.dtypes
# Mostrar los tipos de datos de cada columna en el DataFrame
column_types

## Cargar descripcion de columnas

La siguiente tabla tiene una descripcion del significado de cada columna

In [ ]:
datos_dict = pd.read_excel(
    'https://resources.lendingclub.com/LCDataDictionary.xlsx')
datos_dict.columns = ['feature', 'description']


In [ ]:
datos_dict

### Pickle

Crea codigo para **guardar** y **cargar** el DataFrame de `datos_dict` creada en las celdas anteriores en formato **pickle**

In [ ]:
# Guardar el DataFrame en formato pickle
datos_dict.to_pickle('datos_dict.pkl')

In [ ]:
# Cargar el DataFrame desde el archivo pickle
datos_dict_cargado = pd.read_pickle('datos_dict.pkl')
datos_dict_cargado

## Tipos de Datos

Realiza las transformaciones o casteos (casting) que creas necesarios a tus datos de tal manera que el typo de dato sea adecuado. Al terminar recrea la tabla `column_types` con los nuevos tipos.

No olvides anotar tus justificaciones para recordar cuando te toque explicarlo.

In [ ]:
#Busco que en la descripci?n venga algo que diga date.
def buscar_date(dataframe):
    features_date = set()  

    for index, row in dataframe.iterrows():
        descripcion = row['description']
        feature = row['feature']
        # Verifica que la descripci?n sea una cadena antes de aplicar la expresi?n regular
        if isinstance(descripcion, str) and re.search(r'date', descripcion, flags=re.IGNORECASE):
            # Si se encuentra la palabra "date", agrega el nombre de la feature al conjunto
            features_date.add(feature)

    return features_date
#Obtengo el set con las columas del DataFrame que deben de ser convertidas a date
dates=buscar_date(datos_dict)
#Convierto las columnas correspondientes a date
for columna in dates:
    #Me aseguro que realmente exista la columna
    if columna in loans.columns:
        loans[columna] = pd.to_datetime(loans[columna], errors='coerce', format='%Y-%m-%d')
loans.dtypes

## Manejo de NaNs o missings

Maneja los datos de tipos missing. Elije una estrategia adecuada dependiendo del tipo de dato que le asignaste a la columna.


Crea codigo para **guardar** y **cargar** un archivo JSON en el que se guarde la `estrategia` y `valor` que utilizaste para **imputar**. Por ejemplo: Si hay una columna que se llama `columna 3` y utilizaste la estrategia de imputacion de media, y existe otra llamada `columna 4` y  elegiste la palabra 'missing' el JSON debera contener:  
  
 `{'columna 3':{'estrategia':'mean', 'valor':3.4}, 'columna 4':{'estrategia':'identificador', 'valor':'missing'}}`  

 De tal manera que para cada columna que tenga un metodo de imputacion apunte a otro diccionario donde el **key** `estrategia` describa de manera sencilla el metodo, y el **key** `valor` el valor usado. En general:   
 `{'nombre de la columna':{'estrategia':'descripcion de estrategia', 'valor':'valor utilizado'}}`. 
 

De utilizar mas de un metodo puedes anidarlos en una lista  
  `[{...},{...}]`.  

Incluso si la columna utilizada no sufrio imputacion, es necesario que la agregues al JSON.

La idea es que cualquier otra persona pueda cargar el el archivo JSON con tu funcion, entender que hiciste y replicarlo facilmente. No existe solo una respuesta correcta, pero tendras que justificar y explicar tus deciciones.

### Imputacion

In [ ]:
def imputar_valores_faltantes(dataframe):
        # Elimina filas que contienen solo NaN
    dataframe = dataframe.dropna(axis=0, how='all')

    # Elimina columnas que contienen solo NaN
    dataframe = dataframe.dropna(axis=1, how='all')
    estrategia_imputacion = {}
    for columna in dataframe.columns:
        tipo_dato = dataframe[columna].dtype
        valor = 'la columna no fue modificada'
        estrategia = 'la columna no fue modificada'

        if tipo_dato == 'float64':
            # Con el fin de no cambiar la distribuci?n de los datos
            estrategia = 'mean'
            valor = dataframe[columna].mean()
        else:
            # No tengo la informaci?n suficiente para saber que hacer con los datos
            # de tipo int,date,object,por tanto, es mejor ponerle un missing
            estrategia = 'identificador'
            valor = 'missing' 
        dataframe[columna].fillna(valor, inplace=True)
        estrategia_imputacion[columna] = {'estrategia':estrategia,'valor':valor}
    return dataframe,estrategia_imputacion
loans,estrategia_imputacion = imputar_valores_faltantes(loans)
loans

In [ ]:
estrategia_imputacion

### Codigo para salvar y cargar JSONs

In [ ]:
#Guardamos la informaci?n en el json
with open('estrategia_imputacion.json', 'w') as json_file:
    json.dump(estrategia_imputacion, json_file)

#Cargamos el json que guardamos
with open('estrategia_imputacion.json', 'r') as json_file:
    estrategia_imputacion = json.load(json_file)
estrategia_imputacion